# Final Project: Topic Modeling
**Problem Statement:**

You are to develop your own use case for topic modeling. The use case should involve text data that interests you and should be realistic for a data scientist or machine learning developer. 

In [1]:
import pandas as pd # data manipulation
import langdetect  # language detection
import matplotlib.pyplot  # plotting
import nltk  # natural language processing
import numpy  # arrays and matrices
import pandas  # dataframes
import pyLDAvis  # plotting
import pyLDAvis.lda_model  # plotting
import regex  # regular expressions
import sklearn  # machine learning
import unicodedata  # unicode data manipulation
import random # random number generation

# Text preprocessing and feature extraction
from sklearn.decomposition import NMF  # NMF model
from sklearn.decomposition import LatentDirichletAllocation  # LDA model
from sklearn.feature_extraction.text import TfidfVectorizer # TF-IDF vectorizer
from nltk.stem import WordNetLemmatizer  # lemmatizer
from sklearn.feature_extraction.text import CountVectorizer  # Count vectorizer

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gly3\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gly3\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\gly3\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
url = "https://raw.githubusercontent.com/Hunteracademic/Unsupervised_assignment_1/master/tripadvisor_hotel_reviews.csv"
df = pd.read_csv(url)
df.head()

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5


Executive Summary: In 180 to 200 words, provide an overview of the notebooks you developed. Describe the use case, data, preprocessing steps, model development, and main points of the analysis. State which model works best or that none of the models were satisfactory and provide reasons. Describe the topics and explain how the model will address the use case, or if none of the models worked well, state what the next steps should be.

Preprocessing: Clean and prepare text for LDA and NMF topic modeling. Include steps such as case normalization, lemmatization, stop word removal, and tokenization. 

### Language Filter

In [3]:
def do_language_identifying(txt):
    try: the_language = langdetect.detect(txt)
    except: the_language = 'none'
    return the_language

df['Language'] = df['Review'].apply(do_language_identifying)
df['Language'].value_counts()

Language
en    20475
fr        8
af        5
ca        1
ro        1
nl        1
Name: count, dtype: int64

Removing non-english reviews

In [4]:
reviews_en = df[df['Language'] == 'en']

### Tokenization / Removing Punctuation / Case Normalization

In [5]:
WORD_RE = regex.compile(r"(?V1)\p{L}+(?:[’'-]\p{L}+)*")

def tokenize_for_topics(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = regex.sub(r"[‘’`´]", "'", text)      # normalize apostrophes
    text = regex.sub(r"[‐‑‒–—−]", "-", text)    # normalize dash variants
    return WORD_RE.findall(text)

reviews_en["Tokens"] = reviews_en["Review"].apply(tokenize_for_topics)

reviews_en['Tokens'][0]

['nice',
 'hotel',
 'expensive',
 'parking',
 'got',
 'good',
 'deal',
 'stay',
 'hotel',
 'anniversary',
 'arrived',
 'late',
 'evening',
 'took',
 'advice',
 'previous',
 'reviews',
 'did',
 'valet',
 'parking',
 'check',
 'quick',
 'easy',
 'little',
 'disappointed',
 'non-existent',
 'view',
 'room',
 'room',
 'clean',
 'nice',
 'size',
 'bed',
 'comfortable',
 'woke',
 'stiff',
 'neck',
 'high',
 'pillows',
 'not',
 'soundproof',
 'like',
 'heard',
 'music',
 'room',
 'night',
 'morning',
 'loud',
 'bangs',
 'doors',
 'opening',
 'closing',
 'hear',
 'people',
 'talking',
 'hallway',
 'maybe',
 'just',
 'noisy',
 'neighbors',
 'aveda',
 'bath',
 'products',
 'nice',
 'did',
 'not',
 'goldfish',
 'stay',
 'nice',
 'touch',
 'taken',
 'advantage',
 'staying',
 'longer',
 'location',
 'great',
 'walking',
 'distance',
 'shopping',
 'overall',
 'nice',
 'experience',
 'having',
 'pay',
 'parking',
 'night']

### Removing Stop Words

In [6]:
list_stop_words = nltk.corpus.stopwords.words("English")
reviews_en['Tokens'] = reviews_en['Tokens'].apply(lambda tokens: [token for token in tokens if token not in list_stop_words])

### Lemmatization

In [7]:
lemmatizer = WordNetLemmatizer()
reviews_en['Tokens'] = reviews_en['Tokens'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])

In [8]:
reviews_en['Tokens'].head()

0    [nice, hotel, expensive, parking, got, good, d...
1    [ok, nothing, special, charge, diamond, member...
2    [nice, room, experience, hotel, monaco, seattl...
3    [unique, great, stay, wonderful, time, hotel, ...
4    [great, stay, great, stay, went, seahawk, game...
Name: Tokens, dtype: object

Models: Develop code to first vectorize your text data, and then train at least six LDA and six NMF topic models on these vectors. Use clear section headings for each type of model. Record each set of hyperparameters (for both vectorization and the topic models) that you try, and find the perplexity, word-topic table, and document-topic table for each model. Present this information neatly and use it to select your best LDA and NMF models.

### Vectorizing the text data: LDA
### Hyperparameter Stack:
.....

In [9]:
LDA_data = reviews_en.copy()
LDA_data['clean_text'] = LDA_data['Tokens'].apply(lambda tokens: ' '.join(tokens))
LDA_data[['Tokens', 'clean_text']].head()

,Tokens,clean_text
0,"[nice, hotel, expensive, parking, got, good, d...",nice hotel expensive parking got good deal sta...
1,"[ok, nothing, special, charge, diamond, member...",ok nothing special charge diamond member hilto...
2,"[nice, room, experience, hotel, monaco, seattl...",nice room experience hotel monaco seattle good...
3,"[unique, great, stay, wonderful, time, hotel, ...",unique great stay wonderful time hotel monaco ...
4,"[great, stay, great, stay, went, seahawk, game...",great stay great stay went seahawk game awesom...


In [10]:
number_features = 2000

cv = CountVectorizer(
    max_df=0.95,        
    min_df=2,           
    max_features=number_features
)
 
dtm_cv = cv.fit_transform(LDA_data['clean_text'])


In [11]:
print(dtm_cv[0])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 61 stored elements and shape (1, 2000)>
  Coords	Values
  (0, 1134)	5
  (0, 844)	2
  (0, 618)	1
  (0, 1231)	3
  (0, 752)	1
  (0, 750)	1
  (0, 455)	1
  (0, 1672)	2
  (0, 66)	1
  (0, 89)	1
  (0, 945)	1
  (0, 593)	1
  (0, 1795)	1
  (0, 29)	1
  (0, 1327)	1
  (0, 1460)	1
  (0, 1883)	1
  (0, 303)	1
  (0, 1373)	1
  (0, 552)	1
  (0, 979)	1
  (0, 501)	1
  (0, 1145)	1
  (0, 1892)	1
  (0, 1479)	3
  :	:
  (0, 1099)	1
  (0, 1000)	1
  (0, 517)	1
  (0, 1185)	1
  (0, 796)	1
  (0, 1249)	1
  (0, 1741)	1
  (0, 776)	1
  (0, 1050)	1
  (0, 1144)	1
  (0, 150)	1
  (0, 1343)	1
  (0, 1801)	1
  (0, 1737)	1
  (0, 27)	1
  (0, 1674)	1
  (0, 994)	1
  (0, 989)	1
  (0, 758)	1
  (0, 1912)	1
  (0, 510)	1
  (0, 1565)	1
  (0, 1206)	1
  (0, 619)	1
  (0, 1244)	1


In [12]:
# Initialize the model with default settings
lda_model = LatentDirichletAllocation(n_components=5, random_state=42)

# Fit the model to your Count Vectorized data
lda_model.fit(dtm_cv)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",5
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [13]:
# Get the total number of features (words)
vocab_size = len(cv.get_feature_names_out())
print(f"Vocabulary Size: {vocab_size}")

# Grab 10 random words to verify the Vectorizer worked as expected
print("Random samples from vocabulary:")
feature_names = cv.get_feature_names_out()
for i in range(10):
    random_word_id = random.randint(0, vocab_size - 1)
    print(f"- {feature_names[random_word_id]}")

Vocabulary Size: 2000
Random samples from vocabulary:
- bunch
- class
- dollar
- standard
- accomodations
- home
- young
- occasion
- choice
- flight


In [14]:
# Loop through each topic in lda_model.components_
for index, topic in enumerate(lda_model.components_):
    print(f'THE TOP 10 WORDS FOR TOPIC #{index}')
    
    # Get indices of the top 10 words (highest values in the component array)
    top_word_indices = topic.argsort()[-10:]
    
    # Map indices back to actual words
    top_words = [feature_names[i] for i in top_word_indices]
    print(top_words)
    print('\n')

THE TOP 10 WORDS FOR TOPIC #0
['food', 'breakfast', 'day', 'good', 'service', 'area', 'restaurant', 'room', 'pool', 'hotel']


THE TOP 10 WORDS FOR TOPIC #1
['stayed', 'clean', 'breakfast', 'stay', 'staff', 'good', 'location', 'great', 'room', 'hotel']


THE TOP 10 WORDS FOR TOPIC #2
['told', 'time', 'desk', 'night', 'stay', 'day', 'staff', 'service', 'hotel', 'room']


THE TOP 10 WORDS FOR TOPIC #3
['location', 'stayed', 'nice', 'staff', 'bed', 'night', 'great', 'stay', 'hotel', 'room']


THE TOP 10 WORDS FOR TOPIC #4
['people', 'day', 'pool', 'good', 'time', 'great', 'room', 'food', 'resort', 'beach']




In [15]:
# Generate topic probabilities
topic_results = lda_model.transform(dtm_cv)

# Assign topic index to the dataframe
LDA_data['Topic'] = topic_results.argmax(axis=1)

# Reorder columns and drop 'clean_text' (by omission)
LDA_data = LDA_data[['Review', 'Rating', 'Topic', 'Tokens']]

# Preview results
LDA_data.head()

,Review,Rating,Topic,Tokens
0,nice hotel expensive parking got good deal sta...,4,3,"[nice, hotel, expensive, parking, got, good, d..."
1,ok nothing special charge diamond member hilto...,2,2,"[ok, nothing, special, charge, diamond, member..."
2,nice rooms not 4* experience hotel monaco seat...,3,2,"[nice, room, experience, hotel, monaco, seattl..."
3,"unique, great stay, wonderful time hotel monac...",5,3,"[unique, great, stay, wonderful, time, hotel, ..."
4,"great stay great stay, went seahawk game aweso...",5,2,"[great, stay, great, stay, went, seahawk, game..."


In [16]:
# See the average rating for each topic
print(LDA_data.groupby('Topic')['Rating'].mean())

# See how many reviews fell into each topic
print(LDA_data['Topic'].value_counts())

Topic
0    3.867453
1    4.437922
2    2.624694
3    3.747328
4    3.912162
Name: Rating, dtype: float64
Topic
1    8296
3    5145
4    3552
2    2041
0    1441
Name: count, dtype: int64


### Vectorization for NMF
### Hyperparameter Stack:
.....

In [17]:
# Create a dedicated NMF dataframe from your english reviews
nmf_data = reviews_en.copy()

# Ensure the clean_text column exists (joining the lemmatized tokens)
nmf_data['clean_text'] = nmf_data['Tokens'].apply(lambda tokens: ' '.join(tokens))

# Preview the clean data
nmf_data[['Review', 'clean_text']].head()

,Review,clean_text
0,nice hotel expensive parking got good deal sta...,nice hotel expensive parking got good deal sta...
1,ok nothing special charge diamond member hilto...,ok nothing special charge diamond member hilto...
2,nice rooms not 4* experience hotel monaco seat...,nice room experience hotel monaco seattle good...
3,"unique, great stay, wonderful time hotel monac...",unique great stay wonderful time hotel monaco ...
4,"great stay great stay, went seahawk game aweso...",great stay great stay went seahawk game awesom...


In [21]:
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95,        
    min_df=2,           
    max_features=2000   
    )

dtm_tfidf = tfidf_vectorizer.fit_transform(nmf_data['clean_text'])

In [22]:
print(dtm_tfidf[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 61 stored elements and shape (1, 2000)>
  Coords	Values
  (0, 1134)	0.3053820750271835
  (0, 844)	0.07321626242476674
  (0, 618)	0.10583556919892535
  (0, 1231)	0.35510129245489136
  (0, 752)	0.07891397508907724
  (0, 750)	0.0541240509776907
  (0, 455)	0.11217046814305294
  (0, 1672)	0.10301360372272345
  (0, 66)	0.15410795960382653
  (0, 89)	0.09567674913615373
  (0, 945)	0.11210533830595351
  (0, 593)	0.10538000158258358
  (0, 1795)	0.09707433971669573
  (0, 29)	0.14346472681244288
  (0, 1327)	0.13453262240391622
  (0, 1460)	0.0825932394644516
  (0, 1883)	0.1491904732594534
  (0, 303)	0.08800236453097438
  (0, 1373)	0.12873678980241446
  (0, 552)	0.10252654568977325
  (0, 979)	0.07721130326695999
  (0, 501)	0.11739722102699501
  (0, 1145)	0.1310294245990769
  (0, 1892)	0.07825713083443392
  (0, 1479)	0.10633695247768392
  :	:
  (0, 1099)	0.09138908483336958
  (0, 1000)	0.14158181759965227
  (0, 517)	0.09324661387748452
  (

In [23]:
# Initialize the NMF model
nmf_model = NMF(n_components=5, random_state=42, init='nndsvd')

# Fit the model to your Count Vectorized data
nmf_model.fit(dtm_tfidf)

# Get the vocabulary names
nmf_feature_names = tfidf_vectorizer.get_feature_names_out()

# Display the Top 10 words for each NMF topic
for index, topic in enumerate(nmf_model.components_):
    print(f'THE TOP 10 WORDS FOR NMF TOPIC #{index}')
    top_word_indices = topic.argsort()[-10:]
    print([nmf_feature_names[i] for i in top_word_indices])
    print('\n')

THE TOP 10 WORDS FOR NMF TOPIC #0
['wonderful', 'paris', 'recommend', 'excellent', 'best', 'service', 'stayed', 'stay', 'staff', 'hotel']


THE TOP 10 WORDS FOR NMF TOPIC #1
['drink', 'restaurant', 'good', 'people', 'time', 'day', 'pool', 'food', 'beach', 'resort']


THE TOP 10 WORDS FOR NMF TOPIC #2
['bathroom', 'stay', 'day', 'view', 'check', 'desk', 'floor', 'night', 'bed', 'room']


THE TOP 10 WORDS FOR NMF TOPIC #3
['room', 'nice', 'minute', 'clean', 'hotel', 'location', 'station', 'breakfast', 'walk', 'good']


THE TOP 10 WORDS FOR NMF TOPIC #4
['stayed', 'clean', 'room', 'helpful', 'friendly', 'place', 'staff', 'stay', 'location', 'great']




In [24]:
# Transform the numerical matrix to get topic weights
# This shows how much each document belongs to each of the 5 topics
nmf_topic_results = nmf_model.transform(dtm_tfidf)

# Assign the winning topic index to the dataframe
nmf_data['Topic'] = nmf_topic_results.argmax(axis=1)

# Reorder columns and fix the naming to match your notebook's case-sensitivity
nmf_data = nmf_data[['Review', 'Rating', 'Topic', 'Tokens']]

# Preview results
nmf_data.head()

,Review,Rating,Topic,Tokens
0,nice hotel expensive parking got good deal sta...,4,2,"[nice, hotel, expensive, parking, got, good, d..."
1,ok nothing special charge diamond member hilto...,2,2,"[ok, nothing, special, charge, diamond, member..."
2,nice rooms not 4* experience hotel monaco seat...,3,2,"[nice, room, experience, hotel, monaco, seattl..."
3,"unique, great stay, wonderful time hotel monac...",5,4,"[unique, great, stay, wonderful, time, hotel, ..."
4,"great stay great stay, went seahawk game aweso...",5,2,"[great, stay, great, stay, went, seahawk, game..."


In [25]:
# See the average rating for each topic
print(nmf_data.groupby('Topic')['Rating'].mean())

# See how many reviews fell into each topic
print(nmf_data['Topic'].value_counts()) 

Topic
0    4.320181
1    3.880371
2    3.202498
3    4.172845
4    4.545763
Name: Rating, dtype: float64
Topic
2    5284
3    4235
1    4096
4    3540
0    3320
Name: count, dtype: int64
